In [1]:
import pandas as pd
import ast

from sklearn.model_selection import train_test_split

data_frame = pd.read_csv("cryptonews.csv")
data_frame["sentiment_class"] = data_frame["sentiment"].apply(lambda x: ast.literal_eval(x)['class'])
print(data_frame["sentiment_class"].value_counts())
data_frame = data_frame.drop("url", axis=1)

data_frame["full_text"] = data_frame["title"].fillna('') + " " + data_frame["text"].fillna('')

data_frame = data_frame.sample(frac=1, random_state=42).reset_index(drop=True)
train_df, temp_df = train_test_split(data_frame, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

sentiment_class
positive    13964
neutral     10555
negative     6518
Name: count, dtype: int64
Train: 21725
Validation: 4656
Test: 4656


In [2]:
print(train_df['sentiment_class'].value_counts(normalize=True))
print(val_df['sentiment_class'].value_counts(normalize=True))
print(test_df['sentiment_class'].value_counts(normalize=True))

sentiment_class
positive    0.448884
neutral     0.338734
negative    0.212382
Name: proportion, dtype: float64
sentiment_class
positive    0.444158
neutral     0.350086
negative    0.205756
Name: proportion, dtype: float64
sentiment_class
positive    0.460481
neutral     0.336340
negative    0.203179
Name: proportion, dtype: float64


In [3]:
train_df.to_csv('train.csv', index=False)
val_df.to_csv('val.csv', index=False)
test_df.to_csv('test.csv', index=False)

In [4]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

def preprocess_text(text):
    if not isinstance(text, str):
        if pd.isna(text):
            return ""
        text = str(text)

    text = re.sub(r"http\S+|www\S+|http\S+", '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = text.replace('btc', 'bitcoin').replace('eth', 'ethereum')
    text = text.replace('$', 'dollar')
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()

    stop_words = set(stopwords.words('english'))
    extra_stops = {'said', 'say', 'will', 'one', 'two', 'us', 'new', 'year', 'also', 'today'}
    stop_words = stop_words.union(extra_stops)

    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

In [5]:
train_df["cleaned_data"] = train_df["full_text"].apply(preprocess_text)
val_df["cleaned_data"] = val_df["full_text"].apply(preprocess_text)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

tfidf = TfidfVectorizer(
    max_features=15000,
    stop_words='english',
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=3
)

X_train = tfidf.fit_transform(train_df["cleaned_data"])
y_train = train_df["sentiment_class"]

X_val = tfidf.transform(val_df["cleaned_data"])
y_val = val_df["sentiment_class"]

In [8]:
model = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=1e-5,
    random_state=42,
    max_iter=2000,
    tol=1e-4,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)
val_preds = model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")
print(classification_report(y_val, val_preds))

Validation Accuracy: 0.6460
              precision    recall  f1-score   support

    negative       0.55      0.57      0.56       958
     neutral       0.62      0.65      0.64      1630
    positive       0.71      0.68      0.70      2068

    accuracy                           0.65      4656
   macro avg       0.63      0.63      0.63      4656
weighted avg       0.65      0.65      0.65      4656



In [10]:
joblib.dump(model, 'sentiment_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("save")

save


In [11]:
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

def preprocess_text(text):
    if not isinstance(text, str):
        if pd.isna(text):
            return ""
        text = str(text)

    text = re.sub(r"http\S+|www\S+|http\S+", '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = text.replace('btc', 'bitcoin').replace('eth', 'ethereum')
    text = text.replace('$', 'dollar')
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()

    stop_words = set(stopwords.words('english'))
    extra_stops = {'said', 'say', 'will', 'one', 'two', 'us', 'new', 'year', 'also', 'today'}
    stop_words = stop_words.union(extra_stops)

    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

test_data = pd.read_csv("test.csv")
test_data["cleaned_data"] = test_data["full_text"].apply(preprocess_text)

loaded_model = joblib.load("sentiment_model.pkl")
loaded_tfidf = joblib.load("tfidf_vectorizer.pkl")

X_test = loaded_tfidf.transform(test_data["cleaned_data"])
y_test = test_data["sentiment_class"]

test_pred = loaded_model.predict(X_test)
test_prob = loaded_model.predict_proba(X_test)
test_data["predicted_sentiment"] = test_pred
test_data["prediction_confidence"] = np.max(test_prob, axis=1)

def model_evaluation(y_true, y_pred, model_name="Sentiment Model"):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f})")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
    }

def analyze_predictions(df, text_col="text", pred_col='predicted_sentiment', true_col=None, confidence_col='prediction_confidence'):
    print("Analyze: \n")
    print("top-10 high:\n")
    top_confident = df.nlargest(10, confidence_col)
    for id, row in top_confident.iterrows():
        true_labels = row[true_col] if true_col else "N/A"
        print(f"{row[text_col][:80]}... -> Predict: {row[pred_col]}, "
              f"True: {true_labels}, Confidence: {row[confidence_col]:.4f}")

    print("top-10 low\n")
    low_confident = df.nsmallest(10, confidence_col)
    for idx, row in low_confident.iterrows():
        true_labels = row[true_col] if true_col else "N/A"
        print(f"{row[text_col][:80]}... -> Predict: {row[pred_col]}, "
              f"True: {true_labels}, Confidence: {row[confidence_col]:.4f}")

    print("distribution\n")
    pred_distribution = df[pred_col].value_counts().sort_index()
    for class_label, count in pred_distribution.items():
        percentage = (count / len(df)) * 100
        print(f"Class {class_label}: {count} examples ({percentage:.1f}%)")

if "sentiment_class" in test_data.columns:
    y_true = test_data["sentiment_class"]
    metrics = model_evaluation(y_true, test_pred)
    analyze_predictions(test_data, true_col="sentiment_class")
else:
    print(test_data["prediction_confidence"].describe())
    analyze_predictions(test_data)

print("\n")
for index, row in test_data.head(20).iterrows():
    print(f"{row['text']} -> {row['predicted_sentiment']}")

Accuracy: 0.6433 (64.33)
Precision: 0.6464
Recall: 0.6433
              precision    recall  f1-score   support

    negative       0.57      0.56      0.56       946
     neutral       0.60      0.67      0.63      1566
    positive       0.71      0.66      0.69      2144

    accuracy                           0.64      4656
   macro avg       0.63      0.63      0.63      4656
weighted avg       0.65      0.64      0.64      4656

Analyze: 

top-10 high:

Get your daily, bite-sized digest of cryptoasset and blockchain-related news.... -> Predict: neutral, True: neutral, Confidence: 0.9970
Your daily, bite-sized digest of cryptoasset and blockchain-related news.... -> Predict: neutral, True: neutral, Confidence: 0.9969
Get your daily, bite-sized digest of cryptoasset and blockchain-related news.... -> Predict: neutral, True: neutral, Confidence: 0.9969
Get your daily, bite-sized digest of cryptoasset and blockchain-related news.... -> Predict: neutral, True: neutral, Confidence: 0.9